# Pandas Part 3: Merging DataFrames

Real-world data rarely lives in a single table. You will regularly need to combine a customer table with a transactions table, join product data to sales records, or stack multiple files into one dataset. Pandas provides two primary tools for this: `pd.merge()` for SQL-style joins and `pd.concat()` for stacking DataFrames. Understanding these operations (and knowing which join type to use) is essential for working with realistic datasets.

<div style="background-color:#1e293b;padding:15px;border-left:6px solid #38bdf8;color:#e2e8f0">

<b>What you will learn</b>

- Combine DataFrames with <code>pd.merge()</code>: inner, left, right, and outer joins
- Stack DataFrames vertically with <code>pd.concat()</code>
- Diagnose and fix common merge issues (duplicate rows, missing values after join)

</div>

---
## Step 1: Setup and Sample Data

We will use small hand-crafted DataFrames so you can see exactly what each operation does before applying it to larger datasets.

In [1]:
import pandas as pd

In [2]:
# Employees table
employees = pd.DataFrame({
    'employee_id': [1, 2, 3, 4, 5],
    'name':        ['Alice', 'Bob', 'Carol', 'David', 'Eve'],
    'department_id': [10, 20, 10, 30, 20]
})

print("employees:")
print(employees)

employees:
   employee_id   name  department_id
0            1  Alice             10
1            2    Bob             20
2            3  Carol             10
3            4  David             30
4            5    Eve             20


In [3]:
# Departments table
departments = pd.DataFrame({
    'department_id': [10, 20, 40],
    'department_name': ['Engineering', 'Sales', 'HR']
})

print("departments:")
print(departments)

departments:
   department_id department_name
0             10     Engineering
1             20           Sales
2             40              HR


Notice:
- David (id=4) is in department 30, which does not exist in the departments table
- Department 40 (HR) exists in departments but has no employees

These gaps will produce different results depending on which join type you use.

---
## Step 2: pd.merge() — SQL-Style Joins

<img src="https://www.acuitytraining.co.uk/wp-content/uploads/2022/01/sql-join-types.png.webp" width=400 height=300 />

SQL Join Types. Source: www.acuitytraining.co.uk

### Inner join (default)

Keeps only rows with matching keys in **both** tables. Non-matching rows from either side are dropped.

In [4]:
inner = pd.merge(employees, departments, on='department_id', how='inner')
print("Inner join — only matched rows:")
print(inner)

Inner join — only matched rows:
   employee_id   name  department_id department_name
0            1  Alice             10     Engineering
1            2    Bob             20           Sales
2            3  Carol             10     Engineering
3            5    Eve             20           Sales


David (department 30) is gone — his department_id had no match in `departments`. HR (department 40) is also gone — it had no employees.

### Left join

Keeps **all rows from the left table**. Adds columns from the right table where keys match; fills with `NaN` where they don't.

In [5]:
left = pd.merge(employees, departments, on='department_id', how='left')
print("Left join — all employees kept:")
print(left)

Left join — all employees kept:
   employee_id   name  department_id department_name
0            1  Alice             10     Engineering
1            2    Bob             20           Sales
2            3  Carol             10     Engineering
3            4  David             30             NaN
4            5    Eve             20           Sales


David is now included but `department_name` is `NaN` — his department_id (30) has no match on the right side. This is the most common join in practice: you want to keep all rows from your main table and enrich them with data from a lookup table.

### Right join

Keeps **all rows from the right table**. Mirror image of left join.

In [6]:
right = pd.merge(employees, departments, on='department_id', how='right')
print("Right join — all departments kept:")
print(right)

Right join — all departments kept:
   employee_id   name  department_id department_name
0          1.0  Alice             10     Engineering
1          3.0  Carol             10     Engineering
2          2.0    Bob             20           Sales
3          5.0    Eve             20           Sales
4          NaN    NaN             40              HR


HR (department 40) now appears but `employee_id` and `name` are `NaN`. David's department 30 is missing — it existed in the left table but not the right. Right joins are uncommon; usually the same logic is expressed as a left join with the tables swapped.

> **Note:** `employee_id` shows as `1.0` instead of `1`. When a join introduces `NaN` into an integer column, pandas converts it to float — standard Python integers cannot represent missing values. This is expected behavior and applies to any numeric column that gains NaN through a merge.

### Outer join

Keeps **all rows from both tables**. Fills with `NaN` where there is no match on either side.

In [7]:
outer = pd.merge(employees, departments, on='department_id', how='outer')
print("Outer join — everything:")
print(outer)

Outer join — everything:
   employee_id   name  department_id department_name
0          1.0  Alice             10     Engineering
1          3.0  Carol             10     Engineering
2          2.0    Bob             20           Sales
3          5.0    Eve             20           Sales
4          4.0  David             30             NaN
5          NaN    NaN             40              HR


All 6 rows appear. David gets `NaN` for department_name; HR gets `NaN` for employee_id and name.

### Summary of join types

| Join type | Keeps rows from | Use when |
|---|---|---|
| `inner` | Both tables (matched only) | You only want complete records |
| `left` | All of left + matched right | Main table is left; enriching with lookup |
| `right` | Matched left + all of right | Main table is right (rare — just swap and use left) |
| `outer` | Both tables (all rows) | Auditing what's missing on either side |

### Merging on different column names

When the join key has different names in each table, use `left_on` and `right_on`.

In [8]:
# Let's make Departments table with a differently-named key (for demonstration purposes)
depts_renamed = departments.rename(columns={'department_id': 'dept_id'})

# Now we need to specify the key columns explicitly
result = pd.merge(
    employees,
    depts_renamed,
    left_on='department_id',
    right_on='dept_id',
    how='left'
)
print(result)

   employee_id   name  department_id  dept_id department_name
0            1  Alice             10     10.0     Engineering
1            2    Bob             20     20.0           Sales
2            3  Carol             10     10.0     Engineering
3            4  David             30      NaN             NaN
4            5    Eve             20     20.0           Sales


> **Note:** When using `left_on`/`right_on`, both key columns appear in the result. Drop the redundant one with `.drop()`:

In [9]:
# .drop() — remove a column (or row) from a DataFrame
# columns= takes a single name or a list of names
result_clean = result.drop(columns=['dept_id'])
print("After dropping redundant key column:")
print(result_clean)

After dropping redundant key column:
   employee_id   name  department_id department_name
0            1  Alice             10     Engineering
1            2    Bob             20           Sales
2            3  Carol             10     Engineering
3            4  David             30             NaN
4            5    Eve             20           Sales


---
## Step 3: pd.concat() — Stacking DataFrames

`pd.concat()` stacks DataFrames vertically — adding more rows from one DataFrame below another. It does not perform key-based matching.

In [10]:
# Two batches of salary data from different years
batch_2022 = pd.DataFrame({
    'year': [2022, 2022, 2022],
    'job_title': ['Data Scientist', 'ML Engineer', 'Data Analyst'],
    'salary_usd': [120000, 135000, 95000]
})

batch_2023 = pd.DataFrame({
    'year': [2023, 2023],
    'job_title': ['Data Scientist', 'ML Engineer'],
    'salary_usd': [130000, 145000]
})

print("batch_2022:")
print(batch_2022)
print("\nbatch_2023:")
print(batch_2023)

batch_2022:
   year       job_title  salary_usd
0  2022  Data Scientist      120000
1  2022     ML Engineer      135000
2  2022    Data Analyst       95000

batch_2023:
   year       job_title  salary_usd
0  2023  Data Scientist      130000
1  2023     ML Engineer      145000


In [11]:
# Vertical stack (more rows) — axis=0 (default)
combined = pd.concat([batch_2022, batch_2023], ignore_index=True)
print("Vertical concat:")
print(combined)

Vertical concat:
   year       job_title  salary_usd
0  2022  Data Scientist      120000
1  2022     ML Engineer      135000
2  2022    Data Analyst       95000
3  2023  Data Scientist      130000
4  2023     ML Engineer      145000


> **Note:** `ignore_index=True` resets the index to 0, 1, 2, ... so you do not end up with duplicate index values from the original DataFrames.

In [12]:
# What happens when schemas don't match exactly
batch_2024 = pd.DataFrame({
    'year': [2024],
    'job_title': ['AI Engineer'],
    'salary_usd': [160000],
    'bonus_usd': [20000]  # extra column not in previous batches
})

combined_with_extra = pd.concat([combined, batch_2024], ignore_index=True)
print(combined_with_extra)

   year       job_title  salary_usd  bonus_usd
0  2022  Data Scientist      120000        NaN
1  2022     ML Engineer      135000        NaN
2  2022    Data Analyst       95000        NaN
3  2023  Data Scientist      130000        NaN
4  2023     ML Engineer      145000        NaN
5  2024     AI Engineer      160000    20000.0


Columns that don't exist in a batch are filled with `NaN`. This is expected — you would then handle those missing values as needed.

---
## Step 4: Practical Example — Merging Salaries with a Lookup Table

Let us apply merging to the real salaries dataset to add readable experience level labels.

In [13]:
# Load main dataset
df = pd.read_csv('salaries.csv')

# Create a lookup table for experience level codes
exp_lookup = pd.DataFrame({
    'experience_level': ['EN', 'MI', 'SE', 'EX'],
    'experience_label': ['Junior', 'Mid-level', 'Senior', 'Executive']
})

print("Lookup table:")
print(exp_lookup)

Lookup table:
  experience_level experience_label
0               EN           Junior
1               MI        Mid-level
2               SE           Senior
3               EX        Executive


In [14]:
# Left join to add the label column
df_enriched = pd.merge(df, exp_lookup, on='experience_level', how='left')

print(f"Original shape: {df.shape}")
print(f"Enriched shape: {df_enriched.shape}")  # Same row count
df_enriched[['job_title', 'experience_level', 'experience_label', 'salary_in_usd']].head()

Original shape: (73148, 11)
Enriched shape: (73148, 12)


,job_title,experience_level,experience_label,salary_in_usd
0,Data Product Owner,SE,Senior,170000
1,Data Product Owner,SE,Senior,110000
2,Data Product Owner,SE,Senior,170000
3,Data Product Owner,SE,Senior,110000
4,Engineer,SE,Senior,143000


**TODO:** Create a `company_size_lookup` table that maps `S` → `'Small'`, `M` → `'Medium'`, `L` → `'Large'`. Merge it into `df_enriched` to add a `company_size_label` column. Verify no rows were lost.

In [15]:
# YOUR TURN
company_size_lookup = pd.DataFrame({
    'company_size': ['S', 'M', 'L'],
    'company_size_label': ['Small', 'Medium', 'Large']
})

df_enriched = pd.merge(df_enriched, company_size_lookup, on='company_size', how='left')

print(f"Row count unchanged: {len(df_enriched) == len(df)}")
df_enriched[['company_size', 'company_size_label', 'salary_in_usd']].head()

Row count unchanged: True


,company_size,company_size_label,salary_in_usd
0,M,Medium,170000
1,M,Medium,110000
2,M,Medium,170000
3,M,Medium,110000
4,M,Medium,143000


<details><summary><b>Solution — click to expand</b></summary>

```python
company_size_lookup = pd.DataFrame({
    'company_size': ['S', 'M', 'L'],
    'company_size_label': ['Small', 'Medium', 'Large']
})

df_enriched = pd.merge(df_enriched, company_size_lookup, on='company_size', how='left')

print(f"Row count unchanged: {len(df_enriched) == len(df)}")
df_enriched[['company_size', 'company_size_label', 'salary_in_usd']].head()
```
</details>

---
## Step 5: Diagnosing Merge Issues

Two common problems after a merge:

In [16]:
# Problem 1: Row count unexpectedly increased (many-to-many join)
# This happens when the join key is not unique in the right table

duplicates = pd.DataFrame({
    'experience_level': ['SE', 'SE'],  # duplicate key!
    'label': ['Senior v1', 'Senior v2']
})

result = pd.merge(df.head(5), duplicates, on='experience_level', how='left')
print(f"Input rows: 5, Output rows: {len(result)}")
print(result[['job_title', 'experience_level', 'label']])

Input rows: 5, Output rows: 10
            job_title experience_level      label
0  Data Product Owner               SE  Senior v1
1  Data Product Owner               SE  Senior v2
2  Data Product Owner               SE  Senior v1
3  Data Product Owner               SE  Senior v2
4  Data Product Owner               SE  Senior v1
5  Data Product Owner               SE  Senior v2
6  Data Product Owner               SE  Senior v1
7  Data Product Owner               SE  Senior v2
8            Engineer               SE  Senior v1
9            Engineer               SE  Senior v2


When the right table has duplicate keys, each row in the left table that matches generates one output row per match. **Always verify the row count after a merge** to catch this.

In [17]:
# Problem 2: NaN values after merge (missing matches)
# Check with: result.isnull().sum()

left_result = pd.merge(employees, departments, on='department_id', how='left')
print("NaN values after left join:")
print(left_result.isnull().sum())

# Rows with no match:
print("\nRows with unmatched department:")
print(left_result[left_result['department_name'].isnull()])

NaN values after left join:
employee_id        0
name               0
department_id      0
department_name    1
dtype: int64

Rows with unmatched department:
   employee_id   name  department_id department_name
3            4  David             30             NaN


---
## Summary

You can now:
- Use `pd.merge()` with `how='inner'`, `'left'`, `'right'`, `'outer'` to combine tables on a shared key
- Use `left_on`/`right_on` when key columns have different names, then `.drop()` the redundant one
- Use `pd.concat()` to combine rows from multiple DataFrames into one
- Check row counts and null values after a merge to catch common issues